There are some helper functions that you can use to help integrate your pipeline (or data type) into Cirro.

These include:
- File validation rules / sample matching pattern testing
- Cirro Preprocess script & sample metadata outputs (used for preparing sample sheets for your pipeline)

### File validation rules / sample matching pattern testing

In [1]:
from cirro import DataPortal

portal = DataPortal()
helper = portal.developer_helper

First we need to get a dataset to test against.

In [2]:
dataset = portal.get_dataset(
    project="BTC-GBM-Production",
    dataset="DFCI4.S1.L1.merged-align"
)
files = dataset.list_files()
print(files)

data/DFCI4.S1.L1.merged.aligned-DFCI4-histograms/accuracy.hist (5.04 MB)
data/DFCI4.S1.L1.merged.aligned-DFCI4-histograms/coverage.hist (136.56 KB)
data/DFCI4.S1.L1.merged.aligned-DFCI4-histograms/length.hist (676.47 KB)
data/DFCI4.S1.L1.merged.aligned-DFCI4-histograms/length.unmap.hist (40.70 KB)
data/DFCI4.S1.L1.merged.aligned-DFCI4-histograms/quality.hist (22.27 KB)
data/DFCI4.S1.L1.merged.aligned-DFCI4-histograms/quality.unmap.hist (21.29 KB)
data/DFCI4.S1.L1.merged.aligned-DFCI4.flagstat.tsv (8.03 KB)
data/DFCI4.S1.L1.merged.aligned-DFCI4.sorted.aligned.bam (88.09 GB)
data/DFCI4.S1.L1.merged.aligned-DFCI4.sorted.aligned.bam.bai (53.02 MB)
data/combined_refs.fasta (2.93 GB)
data/combined_refs.fasta.fai (6.33 KB)
data/combined_refs.mmi (6.78 GB)
data/params.json (1.66 KB)
data/versions.txt (140.00 B)
data/wf-alignment-report.html (16.30 MB)


We can see that the dataset has two files. We want to write a regex that will extract the sample name from the file names (4263-B).

Developing a regex can be tricky, [Pythex.org](https://pythex.org/) is a great resource for the regex development since it has a cheat sheet. AI tools are also great for generating regex patterns, but you should always test them to ensure they work as expected.

Name patterns are evaluated in order, so you should put the most specific patterns first.

In [5]:
file_name_patterns = [
    # Illumina format with no lane information
    # Backslashes are escaped with a double backslash, which you would omit in the text field in Cirro
    "(?<sampleName>\\S*)_S(?<libraryIndex>\\S*)_(?<readType>R|I)(?<read>1|2|3|4)_001\\.fastq\\.gz",
    # You can specify multiple patterns if there are different naming conventions
    # Fall back to a more generic pattern if the above does not match
    "(?<sampleName>\\S*)\\.fastq\\.gz"
]

matches = helper.test_file_name_validation_for_dataset(
    project_id=dataset.project_id,
    dataset_id=dataset.id,
    file_name_patterns=file_name_patterns
)

matches.print()

Matches: 0



We can see that it has validated and extracted the sample name from the pattern.
We can now use this pattern when creating the pipeline or data type.

You can also use the `test_file_name_validation` method if you do not have a dataset to test against. This will return a list of matches for the provided file names.

In [6]:
matches = helper.test_file_name_validation(
    file_name_patterns=file_name_patterns,
    file_names=[
        "4263-B_S1_R1_001.fastq.gz",
        "4263-B_S1_R2_001.fastq.gz"
    ]
)
matches.print()


Matches: 2

4263-B_S1_R1_001.fastq.gz
Sample name: 4263-B
Matched regex: (?<sampleName>\S*)_S(?<libraryIndex>\S*)_(?<readType>R|I)(?<read>1|2|3|4)_001\.fastq\.gz

4263-B_S1_R2_001.fastq.gz
Sample name: 4263-B
Matched regex: (?<sampleName>\S*)_S(?<libraryIndex>\S*)_(?<readType>R|I)(?<read>1|2|3|4)_001\.fastq\.gz



### Preprocess testing (sample sheet generation)

To generate the `PreprocessDataset` object using Cirro-provided sample sheets for your pipeline, you can use the `generate_preprocess_for_input_datasets` method.

You can also use `generate_samplesheets_for_dataset` method if you want to access the sample sheets directly. This could be useful if you want to generate test data to write unit tests for your Preprocess script.

In [1]:
dataset = portal.get_dataset(
    project="BTC-GBM-Production",
    dataset="DFCI5.S1.L1.merged-human_variation"
)
files = dataset.list_files()
print(files)

NameError: name 'portal' is not defined

See if my code is right

In [8]:
import json
import pandas as pd
from preprocess_test import generate_mutect2_inputs

# Example: load your process-input.json
with open(".cirro/process-input.json") as f:
    process_inputs = json.load(f)

# Assume ds.files is a DataFrame with ["sample", "file"]
all_inputs = generate_mutect2_inputs(files, process_inputs)

# Check first few inputs
all_inputs[:3]  # preview first 3 sets of inputs

# Optionally, write to JSON files
for i, input_dict in enumerate(all_inputs):
    with open(f"inputs.{i}.json", "w") as f:
        json.dump(input_dict, f, indent=4)


AttributeError: 'DataPortalFiles' object has no attribute 'groupby'

In [25]:
ds = helper.generate_preprocess_for_input_datasets(
    project_id=dataset.project_id,
    input_dataset_ids=[dataset.id])

We can then inspect the `PreprocessDataset` object to see the sample sheets that have been generated.

In [26]:
ds.samplesheet.head()

,sample


In [27]:
ds.files.head()

,sample,file,process,dataset


In [28]:
print(ds.samplesheet.info())
print(ds.files.info())

# Look at the first 10 rows
print(ds.files.head(10))

# Look at the last few rows
print(ds.files.tail())

# Show all unique samples
print(ds.files['sample'].unique())

# Show how many files per sample
print(ds.files['sample'].value_counts())

# Show only the file column for a quick scan
print(ds.files['file'].head(20))

# Check dataset / process metadata
print(ds.files[['dataset', 'process']].drop_duplicates())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   sample  0 non-null      object
dtypes: object(1)
memory usage: 132.0+ bytes
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   sample   0 non-null      object
 1   file     0 non-null      object
 2   process  0 non-null      object
 3   dataset  0 non-null      object
dtypes: object(4)
memory usage: 132.0+ bytes
None
Empty DataFrame
Columns: [sample, file, process, dataset]
Index: []
Empty DataFrame
Columns: [sample, file, process, dataset]
Index: []
[]
Series([], Name: count, dtype: int64)
Series([], Name: file, dtype: object)
Empty DataFrame
Columns: [dataset, process]
Index: []


You can use the two dataframes to create your own sample sheet for your pipeline. See the [Preprocess full example](https://docs.cirro.bio/pipelines/preprocess-script/#full-example) for more details on constructing your sample sheet manually.

The `pivot_samplesheet` method is also available to output the typical sample sheet format used by many pipelines which work on paired-end data.

In [12]:
ds.pivot_samplesheet(
    pivot_columns=['read'],  # Pivot using the read column, to become `fastq_<read>`
    column_prefix='fastq_',  # This is typical of pipelines that work on paired-end data
    metadata_columns=['grouping'],  # My pipeline doesn't allow any additional columns, only `grouping`
    file_filter_predicate='readType == "R"'  # I only want to include read files, not index files
)

UndefinedVariableError: name 'readType' is not defined